# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.metrics import accuracy_score, plot_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.dummy import DummyRegressor
from sklearn.utils import shuffle
from sklearn.tree import plot_tree
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score, GridSearchCV, ParameterGrid
import joblib
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

RS=21

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore).
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [ ]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
df.info()
df

In [ ]:
df2 = pd.read_csv("../data/dayofweek.csv")
df2.info()
df2

In [ ]:
df["dayofweek"] = df2["dayofweek"]
df

In [ ]:
target = df["dayofweek"]
features = df.drop(columns=["dayofweek"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, 
                                                    random_state=21, stratify=target)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
model = svm.SVC(random_state=21, probability=True)

parametrs = {
                'kernel': ("linear", "rbf", "sigmoid"),
                'C': (0.01, 0.1, 1, 1.5, 5, 10),
                'gamma': ("scale", "auto"),
                'class_weight': ("balanced", None)
            }

grid = GridSearchCV(model, parametrs, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
print(grid.best_params_)
print('accuracy final {:.2f} %'.format(grid.best_score_))
print(grid.best_score_)

In [ ]:
df_grid = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
df_grid

## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
model = DecisionTreeClassifier(random_state=21)
parametrs = {
                'max_depth': range(1, 50),
                'criterion': ("entropy", "gini"),
                'class_weight': ("balanced", None)
            }

grid = GridSearchCV(model, parametrs, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
print(grid.best_params_)
print('accuracy final {:.2f} %'.format(grid.best_score_))
print(grid.best_score_)

In [ ]:
df_grid = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
df_grid

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
model = RandomForestClassifier(random_state=21)
parametrs = {
                'n_estimators': (5, 10, 50, 100),
                'max_depth': range(1, 50),
                'criterion': ("entropy", "gini"),
                'class_weight': ("balanced", None)
            }

grid = GridSearchCV(model, parametrs,scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
print(grid.best_params_)
print('accuracy final {:.2f} %'.format(grid.best_score_))
print(grid.best_score_)

In [ ]:
df_grid = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
df_grid

## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
parametrs["random_state"] = (21,)
parametrs

In [ ]:
params = list(ParameterGrid(parametrs))
params 

In [ ]:
def ft_mygrid_search(params):
    data = []
    for param in tqdm(params):
        row = param
        model = RandomForestClassifier(**param)
        scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
        row['mean_accuracy'] = np.array(scores).mean()
        row['std_accuracy'] = np.array(scores).std()
        data.append(row)
    return pd.DataFrame(data).sort_values("mean_accuracy", ascending=False)

In [ ]:
df_mygrid = ft_mygrid_search(params)
df_mygrid

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [ ]:
model = RandomForestClassifier(random_state=21, class_weight=None, criterion="gini",
                               max_depth=28, n_estimators=50)
model.fit(X_train, y_train)
predict = model.predict(X_test)
accuracy_score(y_test, predict)